In [6]:
# ============================================================
# NOTEBOOK 6 — DEMOGRAPHIC BANDS
# Add region, age_group, income_group, household_group, 
# and demographic_profile to the customer-level table.
# ============================================================

In [7]:
import pandas as pd
import numpy as np
import os

In [8]:
# Assigning path
path = r'/Users/elia/Desktop/[02] Data Projects/[2] Working Folder/[3] InstaCart Store'

In [9]:
# ------------------------------------------------------------
# LOAD CHECKPOINTS
# ------------------------------------------------------------
df_customers_agg = pd.read_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'customers_agg_step5.pkl')
)
print(f"Customer aggregation loaded: {df_customers_agg.shape}")

Customer aggregation loaded: (206209, 14)


In [11]:
# We need the demographic source columns from the master file.
# We pull only one row per user (demographics are constant per user)
# to avoid bringing in 32M rows.

In [12]:
df_master = pd.read_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'instacart_master.pkl')
)
print(f"Master loaded: {df_master.shape}")

demographic_cols = ['user_id', 'gender', 'state', 'age',
                    'date_joined', 'n_dependants', 'fam_status', 'income']
df_demo = df_master[demographic_cols].drop_duplicates(subset=['user_id']).reset_index(drop=True)

assert df_demo.shape[0] == 206209, f"Demographic source has wrong row count: {df_demo.shape[0]}"
assert df_demo['user_id'].duplicated().sum() == 0
print(f"✅ Demographic source extracted: {df_demo.shape}")
print()

Master loaded: (32398592, 21)
✅ Demographic source extracted: (206209, 8)



In [13]:
# ------------------------------------------------------------
# 6A — MERGE DEMOGRAPHICS ONTO CUSTOMER TABLE
# ------------------------------------------------------------

In [14]:
rows_before = df_customers_agg.shape[0]

df_customers_agg = df_customers_agg.merge(df_demo, on='user_id', how='left')

assert df_customers_agg.shape[0] == rows_before, "Merge altered row count"
assert df_customers_agg.isnull().sum().sum() == 0, "Merge produced nulls"

print(f"✅ Demographics merged: {df_customers_agg.shape}")
print()

✅ Demographics merged: (206209, 21)



In [15]:
# ------------------------------------------------------------
# 6B — REGION (from state)
# ------------------------------------------------------------
# US Census Bureau 4-region classification.
# Exhaustive: every US state + DC is mapped.

In [16]:
region_map = {
    # Northeast
    'Maine': 'Northeast', 'New Hampshire': 'Northeast', 'Vermont': 'Northeast',
    'Massachusetts': 'Northeast', 'Rhode Island': 'Northeast', 'Connecticut': 'Northeast',
    'New York': 'Northeast', 'Pennsylvania': 'Northeast', 'New Jersey': 'Northeast',
    # Midwest
    'Wisconsin': 'Midwest', 'Michigan': 'Midwest', 'Illinois': 'Midwest',
    'Indiana': 'Midwest', 'Ohio': 'Midwest', 'North Dakota': 'Midwest',
    'South Dakota': 'Midwest', 'Nebraska': 'Midwest', 'Kansas': 'Midwest',
    'Minnesota': 'Midwest', 'Iowa': 'Midwest', 'Missouri': 'Midwest',
    # South
    'Delaware': 'South', 'Maryland': 'South', 'District of Columbia': 'South',
    'Virginia': 'South', 'West Virginia': 'South', 'North Carolina': 'South',
    'South Carolina': 'South', 'Georgia': 'South', 'Florida': 'South',
    'Kentucky': 'South', 'Tennessee': 'South', 'Mississippi': 'South',
    'Alabama': 'South', 'Oklahoma': 'South', 'Texas': 'South',
    'Arkansas': 'South', 'Louisiana': 'South',
    # West
    'Idaho': 'West', 'Montana': 'West', 'Wyoming': 'West', 'Nevada': 'West',
    'Utah': 'West', 'Colorado': 'West', 'Arizona': 'West', 'New Mexico': 'West',
    'Alaska': 'West', 'Washington': 'West', 'Oregon': 'West',
    'California': 'West', 'Hawaii': 'West'
}

df_customers_agg['region'] = df_customers_agg['state'].map(region_map)

In [17]:
# Hard assertion: every state must map to a region
assert df_customers_agg['region'].isnull().sum() == 0, \
    f"Unmapped states: {df_customers_agg.loc[df_customers_agg['region'].isnull(), 'state'].unique()}"

In [18]:
# Set as ordered categorical
df_customers_agg['region'] = pd.Categorical(
    df_customers_agg['region'],
    categories=['Northeast', 'Midwest', 'South', 'West'],
    ordered=True
)

print("✅ Region assigned")
print(df_customers_agg['region'].value_counts().sort_index().to_string())
print()

✅ Region assigned
region
Northeast    36388
Midwest      48519
South        68737
West         52565



In [19]:
# ------------------------------------------------------------
# 6C — AGE GROUP
# ------------------------------------------------------------
# Bin edges: [-inf, 30, 45, 60, +inf]
# right=False means each bin is [lower, upper):
#   [-inf, 30) → Young Adults  (18–29)
#   [ 30, 45) → Adults         (30–44)
#   [ 45, 60) → Middle-Aged    (45–59)
#   [ 60, +inf) → Older Adults (60+)

In [20]:
age_bins   = [-np.inf, 30, 45, 60, np.inf]
age_labels = ['Young Adults', 'Adults', 'Middle-Aged', 'Older Adults']

df_customers_agg['age_group'] = pd.cut(
    df_customers_agg['age'],
    bins=age_bins,
    labels=age_labels,
    right=False,
    include_lowest=True
)
df_customers_agg['age_group'] = df_customers_agg['age_group'].cat.set_categories(
    age_labels, ordered=True
)

assert df_customers_agg['age_group'].isnull().sum() == 0
print("✅ Age group assigned")
print(df_customers_agg['age_group'].value_counts().sort_index().to_string())
print()

✅ Age group assigned
age_group
Young Adults    38668
Adults          48232
Middle-Aged     48582
Older Adults    70727



In [21]:
# ------------------------------------------------------------
# 6D — INCOME GROUP
# ------------------------------------------------------------
# Bin edges: [-inf, 50_000, 100_000, 150_000, +inf]

In [22]:
income_bins   = [-np.inf, 50_000, 100_000, 150_000, np.inf]
income_labels = ['Low Income', 'Middle Income', 'High Income', 'Very High Income']

df_customers_agg['income_group'] = pd.cut(
    df_customers_agg['income'],
    bins=income_bins,
    labels=income_labels,
    right=False,
    include_lowest=True
)
df_customers_agg['income_group'] = df_customers_agg['income_group'].cat.set_categories(
    income_labels, ordered=True
)

assert df_customers_agg['income_group'].isnull().sum() == 0
print("✅ Income group assigned")
print(df_customers_agg['income_group'].value_counts().sort_index().to_string())
print()

✅ Income group assigned
income_group
Low Income          34105
Middle Income       84846
High Income         64004
Very High Income    23254



In [30]:
# ------------------------------------------------------------
# 6E — HOUSEHOLD GROUP
# ------------------------------------------------------------
# Four exhaustive categories matching the actual source data.
# This dataset rigidly pairs (fam_status, n_dependants):
#   single             → always 0 deps   → Single No Kids
#   married            → always 1+ deps  → Family Household
#   divorced/widowed   → always 0 deps   → Divorced/Widowed
#   living with parents and siblings → always 1+ deps → Living with Family
# Two theoretically-possible combinations (single parent, childless
# couple) do not exist in the source. We do not include empty
# categories — better to reflect what's actually there.

In [31]:
def assign_household(row):
    """
    Map (fam_status, n_dependants) to a household group.
    Four categories cover every observed combination.
    """
    fs = row['fam_status']
    
    if fs == 'single':
        return 'Single No Kids'
    if fs == 'married':
        return 'Family Household'
    if fs == 'divorced/widowed':
        return 'Divorced/Widowed'
    if fs == 'living with parents and siblings':
        return 'Living with Family'
    return None  # caller will assert no nulls

df_customers_agg['household_group'] = df_customers_agg.apply(assign_household, axis=1)

assert df_customers_agg['household_group'].isnull().sum() == 0, \
    "Unexpected fam_status values produced unmapped households"

household_labels = [
    'Single No Kids', 'Family Household',
    'Divorced/Widowed', 'Living with Family'
]

df_customers_agg['household_group'] = pd.Categorical(
    df_customers_agg['household_group'],
    categories=household_labels,
    ordered=True
)

print("✅ Household group assigned")
print(df_customers_agg['household_group'].value_counts().sort_index().to_string())
print()

✅ Household group assigned
household_group
Single No Kids         33962
Family Household      144906
Divorced/Widowed       17640
Living with Family      9701



In [32]:
# ------------------------------------------------------------
# 6F — DEMOGRAPHIC PROFILE (concatenated string)
# ------------------------------------------------------------
# Useful for Power BI single-axis slicing. Format matches the
# old export convention: "age_group | income_group | household_group"

In [33]:
df_customers_agg['demographic_profile'] = (
    df_customers_agg['age_group'].astype(str)       + ' | ' +
    df_customers_agg['income_group'].astype(str)    + ' | ' +
    df_customers_agg['household_group'].astype(str)
)

print("✅ Demographic profile assembled")
print(f"   Unique profiles: {df_customers_agg['demographic_profile'].nunique()}")
print(f"   Sample top profiles:")
print(df_customers_agg['demographic_profile'].value_counts().head(8).to_string())
print()

✅ Demographic profile assembled
   Unique profiles: 36
   Sample top profiles:
demographic_profile
Older Adults | High Income | Family Household         25219
Adults | Middle Income | Family Household             22093
Middle-Aged | High Income | Family Household          17172
Young Adults | Middle Income | Family Household       14455
Older Adults | Middle Income | Family Household       11750
Older Adults | Very High Income | Family Household     9092
Older Adults | High Income | Divorced/Widowed          8358
Middle-Aged | Middle Income | Family Household         8034



In [34]:
# ------------------------------------------------------------
# 6G — FINAL VALIDATION
# ------------------------------------------------------------

In [35]:
assert df_customers_agg.shape[0] == 206209
assert df_customers_agg['user_id'].duplicated().sum() == 0

# All four new categorical columns must be fully populated
for col in ['region', 'age_group', 'income_group', 'household_group',
            'demographic_profile']:
    assert df_customers_agg[col].isnull().sum() == 0, f"{col} has nulls"

# Number of unique demographic profiles cannot exceed
# 4 (age) × 4 (income) × 4 (household) = 64
assert df_customers_agg['demographic_profile'].nunique() <= 64

print("✅ All Notebook 6 assertions passed")
print()

✅ All Notebook 6 assertions passed



In [36]:
# ------------------------------------------------------------
# 6H — SAVE CHECKPOINT
# ------------------------------------------------------------

In [37]:
df_customers_agg.to_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'customers_agg_step6.pkl')
)

print(f"✅ NOTEBOOK 6 COMPLETE — checkpoint saved: customers_agg_step6.pkl")
print(f"   Shape: {df_customers_agg.shape}")
print(f"   Columns: {df_customers_agg.columns.tolist()}")

✅ NOTEBOOK 6 COMPLETE — checkpoint saved: customers_agg_step6.pkl
   Shape: (206209, 26)
   Columns: ['user_id', 'unique_orders', 'unique_products', 'avg_product_price', 'total_reordered_items', 'total_line_items', 'reorder_rate', 'orders_pct_rank', 'reorder_pct_rank', 'products_pct_rank', 'price_pct_rank', 'behaviour_score', 'priority_level', 'behaviour_segment', 'gender', 'state', 'age', 'date_joined', 'n_dependants', 'fam_status', 'income', 'region', 'age_group', 'income_group', 'household_group', 'demographic_profile']


In [38]:
# Maintain-only distribution analysis
maintain = df_customers_agg[df_customers_agg['priority_level'] == 'Maintain'].copy()

print(f"Maintain tier size: {maintain.shape[0]:,}")
print()

print("Price percentile rank distribution within Maintain:")
print(maintain['price_pct_rank'].describe(percentiles=[.1, .25, .5, .75, .9]).round(3))
print()

print("Orders percentile rank distribution within Maintain:")
print(maintain['orders_pct_rank'].describe(percentiles=[.1, .25, .5, .75, .9]).round(3))
print()

# Check candidate sub-segment sizes for several threshold options
print("Sub-segment size simulation:")
print()
for price_thr in [0.60, 0.65, 0.70, 0.75]:
    premium = (maintain['price_pct_rank'] >= price_thr).sum()
    pct = premium / len(maintain) * 100
    print(f"  Premium (price >= {price_thr}): {premium:>6,} customers ({pct:5.1f}%)")
print()

for orders_thr, price_thr in [(0.40, 0.40), (0.50, 0.40), (0.50, 0.50), (0.60, 0.40)]:
    upsell = ((maintain['orders_pct_rank'] >= orders_thr) & 
              (maintain['price_pct_rank'] < price_thr)).sum()
    pct = upsell / len(maintain) * 100
    print(f"  Upsell (orders >= {orders_thr}, price < {price_thr}): {upsell:>6,} customers ({pct:5.1f}%)")
print()

# Simulate the trio with the originally proposed thresholds
premium    = (maintain['price_pct_rank'] >= 0.70).sum()
upsell     = ((maintain['orders_pct_rank'] >= 0.50) & (maintain['price_pct_rank'] < 0.40)).sum()
stable     = len(maintain) - premium - upsell
print("With proposed thresholds (price>=0.70, orders>=0.50 & price<0.40):")
print(f"  Premium Higher Price Buyers:      {premium:>6,} ({premium/len(maintain)*100:5.1f}%)")
print(f"  Upsell Frequent Low-Price Buyers: {upsell:>6,} ({upsell/len(maintain)*100:5.1f}%)")
print(f"  Maintain Stable Mid-Value:        {stable:>6,} ({stable/len(maintain)*100:5.1f}%)")

Maintain tier size: 54,017

Price percentile rank distribution within Maintain:
count    54017.000
mean         0.502
std          0.289
min          0.000
10%          0.103
25%          0.252
50%          0.502
75%          0.754
90%          0.902
max          1.000
Name: price_pct_rank, dtype: float64

Orders percentile rank distribution within Maintain:
count    54017.000
mean         0.502
std          0.121
min          0.058
10%          0.323
25%          0.438
50%          0.485
75%          0.594
90%          0.649
max          0.934
Name: orders_pct_rank, dtype: float64

Sub-segment size simulation:

  Premium (price >= 0.6): 21,778 customers ( 40.3%)
  Premium (price >= 0.65): 19,163 customers ( 35.5%)
  Premium (price >= 0.7): 16,422 customers ( 30.4%)
  Premium (price >= 0.75): 13,692 customers ( 25.3%)

  Upsell (orders >= 0.4, price < 0.4): 18,404 customers ( 34.1%)
  Upsell (orders >= 0.5, price < 0.4): 13,167 customers ( 24.4%)
  Upsell (orders >= 0.5, price < 0.5): 